# Fuel Price UK

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as np
import warnings
warnings.filterwarnings('ignore') # HIdes harmless warning clutter


In [2]:
#Load the dataset

data = pd.read_csv("../data/price_history.csv")

In [3]:
#checking row and columns
data.shape 

(229232, 6)

In [4]:
#Checking the column names
data.columns

Index(['id', 'node_id', 'fuel_type', 'price_pence', 'recorded_at',
       'source_updated_at'],
      dtype='object')

In [5]:
# Checking the first 5 data
data.head()

,id,node_id,fuel_type,price_pence,recorded_at,source_updated_at
0,1,4882e3fee979cfefa29a8777ce57ca0ecea27b4f12ba7f...,E10,126.0,2026-02-07T21:40:01.511Z,2026-02-06T12:46:05.000Z
1,2,4882e3fee979cfefa29a8777ce57ca0ecea27b4f12ba7f...,B7_STANDARD,135.0,2026-02-07T21:40:01.511Z,2026-02-06T12:46:05.000Z
2,3,b739362af81acc9fec9eda6f155348125fa2d5c1772c96...,E10,127.0,2026-02-07T21:40:01.511Z,2026-02-01T11:04:45.000Z
3,4,b739362af81acc9fec9eda6f155348125fa2d5c1772c96...,B7_STANDARD,138.0,2026-02-07T21:40:01.511Z,2026-02-01T11:04:45.000Z
4,5,ae5b2f8f6979b8b2460ff52f85619dd0dfc6a10bc98ca1...,E5,146.0,2026-02-07T21:40:01.511Z,2026-02-06T13:57:50.000Z


In [6]:
#chekcing the bottom 5 data
data.tail()

,id,node_id,fuel_type,price_pence,recorded_at,source_updated_at
229227,229772,b4d4eaa2aaf0a5617a5e4d467bd65513bde7b4ccbc7c0f...,E10,149.9,2026-03-22T23:28:06.115Z,2026-03-22T23:25:29.000Z
229228,229773,b4d4eaa2aaf0a5617a5e4d467bd65513bde7b4ccbc7c0f...,B7_STANDARD,173.9,2026-03-22T23:28:06.115Z,2026-03-22T23:25:29.000Z
229229,229774,b4d4eaa2aaf0a5617a5e4d467bd65513bde7b4ccbc7c0f...,B7_PREMIUM,189.9,2026-03-22T23:28:06.115Z,2026-03-22T23:25:29.000Z
229230,229775,62934f040255eac8763683d845aa0daa974bba7018a1b4...,E10,146.9,2026-03-22T23:28:06.115Z,2026-03-22T23:26:45.000Z
229231,229776,62934f040255eac8763683d845aa0daa974bba7018a1b4...,B7_STANDARD,169.9,2026-03-22T23:28:06.115Z,2026-03-22T23:26:45.000Z


In [7]:
# Understanding the data type
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 229232 entries, 0 to 229231
Data columns (total 6 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   id                 229232 non-null  int64  
 1   node_id            229232 non-null  object 
 2   fuel_type          229232 non-null  object 
 3   price_pence        229232 non-null  float64
 4   recorded_at        229232 non-null  object 
 5   source_updated_at  229232 non-null  object 
dtypes: float64(1), int64(1), object(4)
memory usage: 10.5+ MB


Here, the recorded_at and source_updated_at should be Date and time not object

In [8]:
# Fixing the date column in recorded_at

data['recorded_at'] = pd.to_datetime(data['recorded_at'])
data['source_updated_at'] = pd.to_datetime(data['source_updated_at'])

# Extracting the useful data type only 
data['date'] = data['recorded_at'].dt.date
data['hour'] = data['recorded_at'].dt.hour
data['day_of_week'] = data['recorded_at'].dt.day_name()
data['week'] = data['recorded_at'].dt.isocalendar().week


In [9]:
print("Date range of this dataset")
print(f"From:{data['recorded_at'].min()}")
print(f"From:{data['recorded_at'].max()}")

Date range of this dataset
From:2026-02-07 21:40:01.511000+00:00
From:2026-03-22 23:28:06.115000+00:00


This data covers roughly 44 days about 6 weeks of UK fuel prices across 7,442 stations.

In [10]:
# Check data quality
data.isnull().sum()

id                   0
node_id              0
fuel_type            0
price_pence          0
recorded_at          0
source_updated_at    0
date                 0
hour                 0
day_of_week          0
week                 0
dtype: int64

In [11]:
#Checking the price statistics
data['price_pence'].describe()

count    229232.000000
mean        153.733196
std          41.009238
min           0.000000
25%         140.000000
50%         149.900000
75%         163.900000
max        1949.000000
Name: price_pence, dtype: float64

1949 penny is so extreme it drags the mean upward for that fuel type. which cause wrong insight 

In [12]:
print("\n- SUSPICIOUS PRICES -")
print(f"Prices equal to 0 penny :{(data['price_pence'] == 0).sum()}")
print(f"Prices below 50 penny :{(data['price_pence'] < 50).sum()}")
print(f"Prices above 500 penny :{(data['price_pence'] > 500).sum()}")



- SUSPICIOUS PRICES -
Prices equal to 0 penny :2
Prices below 50 penny :494
Prices above 500 penny :327


- When Fuel Price is at 0 Penny

 A fuel price of zero penny is physically impossible. You cannot buy fuel for free. These are almost certainly one of two things: a system error where the station's pump failed to transmit a price, or a placeholder value inserted when no data was available.

- When Fule Price is at 50 penny

UK fuel has never been below 50p per litre in modern history. So these 494 records are not real prices. They likely come from stations entering prices in the wrong unit (pounds instead of pence), test entries from engineers setting up new stations, or corrupted transmissions. 

- When Fule Price is at 500 Penny

500p per litre would be £5.00 per litre — roughly 3x the real price. These are the opposite problem: values that are wildly too high. Same causes — unit errors, data corruption, or test data that never got cleaned out of the production system.

In [13]:
# Removing the bad data 

data_clean = data[data['price_pence']>=50 & (data['price_pence']<=400)].copy()
removed = len(data)-len(data_clean)

print(f"ORiginal rows:{len(data):,}")
print(f"Removed Rows:{removed:,}({removed/len(data)*100:.2f}% of the data)")
print(f"Clean rows:{len(data_clean):,}")
print(f"\nPrice range after cleaning:")
print(f"  Min: {data_clean['price_pence'].min():.1f}p")
print(f"  Max: {data_clean['price_pence'].max():.1f}p")
print(f"  Avg: {data_clean['price_pence'].mean():.1f}p")

ORiginal rows:229,232
Removed Rows:0(0.00% of the data)
Clean rows:229,232

Price range after cleaning:
  Min: 0.0p
  Max: 1949.0p
  Avg: 153.7p


why 50p and 400p > because real UK fuel prices in early 2026 sit in that band . anything price beside that are consider as data entry mistake 

In [14]:
# Cell 8 - Average price by fuel type
avg_by_fuel = data_clean.groupby('fuel_type')['price_pence'].agg(
     count='count',
     mean_price='mean',
     min_price='min',
     max_price='max'
).round(2).sort_values('mean_price', ascending=False)

print("AVERAGE PRICE BY FUEL TYPE (pence per litre)\n")
print(avg_by_fuel.to_string())

AVERAGE PRICE BY FUEL TYPE (pence per litre)

             count  mean_price  min_price  max_price
fuel_type                                           
B7_PREMIUM   35334      173.94       1.00     1949.0
HVO            321      165.61      11.61      444.0
E5           44064      157.48       0.00     1709.0
B7_STANDARD  80203      154.35       1.00     1777.0
B10            353      142.37       1.00      921.0
E10          68957      140.27       1.00     1479.0


max price can't be that high for all the fule price . it might be data entry mistake .

In [15]:
# Checking the exterme price properly 
print("Heighest prices in dataset \n")
print(data.nlargest(10,'price_pence')[['node_id','fuel_type','price_pence','recorded_at']])

Heighest prices in dataset 

                                                  node_id    fuel_type  \
130059  7e54b71d4fc9e3cd9d0f87e6ff69184eb7001a3de05c08...   B7_PREMIUM   
212282  0c5dee22ecf242d1027b6118eac744e01755e080df95a1...  B7_STANDARD   
169547  3631dbafa225c68604b9d288cf0b44e26a14912ce713d8...  B7_STANDARD   
210995  91909366652067d23e82c14e9d55924d3ddf1549434440...  B7_STANDARD   
130056  7e54b71d4fc9e3cd9d0f87e6ff69184eb7001a3de05c08...           E5   
3311    24c76ae2c22792c298a81faeccbf2874cf2e9549135fcf...           E5   
130058  7e54b71d4fc9e3cd9d0f87e6ff69184eb7001a3de05c08...  B7_STANDARD   
137760  fa083c8f11b40cbf72156baaa440be52c841e40b51a637...  B7_STANDARD   
186125  d1496354a7989d8aaf8bc0dbe3cc3331c4d677dcd85544...  B7_STANDARD   
2919    05c79b55166e623e0f9f4de82fc6f00dd31329fc85a5ba...  B7_STANDARD   

        price_pence                      recorded_at  
130059       1949.0 2026-03-09 12:48:07.713000+00:00  
212282       1777.0 2026-03-20 18:12:06.829000

In [16]:
#chekcing the exterme records per fule type only 

print("Exterme recoreds per fule type\n ")
extreme_high = data[data['price_pence']>500]
print(extreme_high.groupby('fuel_type')['price_pence'].agg(['count','mean','max']))

Exterme recoreds per fule type
 
             count         mean     max
fuel_type                              
B10              2   722.450000   921.0
B7_PREMIUM      15  1285.386667  1949.0
B7_STANDARD     44  1352.287727  1777.0
E10            137  1062.823358  1479.0
E5             129  1063.530233  1709.0


In [17]:
# chekcing the price records per fule type below 50 penny 
print("records per fule type(belwo 50 penny)")
exterme_low = data[data['price_pence']<50]
print(exterme_low.groupby('fuel_type')['price_pence'].agg(['count','mean','max']))

records per fule type(belwo 50 penny)
             count       mean    max
fuel_type                           
B10             45  17.205556  31.31
B7_PREMIUM      67   3.970746  19.30
B7_STANDARD    135   7.428667  41.41
E10            136   3.746397  39.90
E5              97   7.590000  21.21
HVO             14  17.181429  17.61


In [18]:
#Does same stations have multiple bad price?

print("do same station have multiple bad prices ")
bad_stations = data[data['price_pence']>500]['node_id'].value_counts().head(10)
print(bad_stations)

do same station have multiple bad prices 
node_id
c9aa1695a016aa511c09299537f3c6536c428a1958eacaa21f2b545f455ecdb5    215
0eab9305ef5cfedf81d5b430c2050f50882997a00abc926d11a73ea36e5fe398      6
eb2a4ec0d0d85fbe3f6ea05566415571be7274212e00e16c2d003239e7930cf2      4
32e5e276d604141ace5e2af9358d1ba0d16cf0d31566f0e3e86c4fafb82b9c3e      4
9a8f20268e424c379929cbb3cf83925a2ab1b7dda461795f46a164192fd576aa      4
fdf4974536725227197fa9209fcfc82f234cfe342613b63cf72a100ec1dec152      4
523e6288311b2a2437084e803bd869932ba9969c7bcacf4aa11bee2e98f630a9      4
3c8b86979ee1c44d3b365287150ed428293c2190f349b9af6a4f595bcfe1dc95      4
8cc27e4c09366d7f6fc648c6c6d4eda8e069a70254c6687b291959417f57412d      4
7e54b71d4fc9e3cd9d0f87e6ff69184eb7001a3de05c085565c92028a6f368ef      4
Name: count, dtype: int64


one station has 215 bad price records all by itself. every other station has only 4 - 6 . that single staion account tha more than 65% of all bad high price data out of 327 set. this might not be a random data entry mistake. it might be the cause of broken entyr sensor at that specifc staition that has been consistently transmitting garbage price .


In [25]:
# Deep dive into the problem station 

problem_station = 'c9aa1695a016aa511c09299537f3c6536c428a1958eacaa21f2b545f455ecdb5'

station_data = data[data['node_id']==problem_station]
print(f"total records for this stations: {len(station_data)}")
print(f"bad records (> 500penny):{len(station_data[station_data['price_pence']>500])}") # 500 penny vanda dheari liney petrol ko 
print(f"Good records (< 500penny):{len(station_data[station_data['price_pence']<500])}") # 500 penny vanda kaam liney



total records for this stations: 432
bad records (> 500penny):215
Good records (< 500penny):217


In [ ]:
#Checking the fuel type which it sellls
station_data['fuel_type'].value_counts() #high price ma petrol dine station ko type of fuel 

fuel_type
E5             216
E10            213
B7_STANDARD      3
Name: count, dtype: int64

In [27]:
# Sample of bad prices 
bad_price = station_data[station_data['price_pence']>500][['fuel_type','price_pence','recorded_at']]
print(bad_price.sort_values('recorded_at').head(15))

      fuel_type  price_pence                      recorded_at
8506         E5       1000.0 2026-02-07 21:40:02.318000+00:00
8507        E10       1000.0 2026-02-07 21:40:02.318000+00:00
28018        E5       1000.0 2026-02-17 08:10:32.378000+00:00
28062        E5       1000.0 2026-02-17 08:57:03.384000+00:00
28078       E10       1000.0 2026-02-17 09:10:02.240000+00:00
28380        E5       1000.0 2026-02-17 11:52:34.734000+00:00
28473       E10       1000.0 2026-02-17 12:16:38.550000+00:00
28599        E5       1000.0 2026-02-17 12:40:53.969000+00:00
28614       E10       1000.0 2026-02-17 12:44:15.513000+00:00
28700        E5       1000.0 2026-02-17 13:25:11.607000+00:00
28778       E10       1000.0 2026-02-17 14:40:38.695000+00:00
29279        E5       1000.0 2026-02-17 17:53:07.771000+00:00
29576       E10       1000.0 2026-02-18 08:49:01.847000+00:00
29603        E5       1000.0 2026-02-18 09:52:08.104000+00:00
29751       E10       1000.0 2026-02-18 12:00:50.487000+00:00


In [29]:
#Sample of good price 
good_price = station_data[station_data['price_pence']<500][['fuel_type','price_pence','recorded_at']]
print(good_price.sort_values('recorded_at').head(15))

      fuel_type  price_pence                      recorded_at
27920        E5        124.0 2026-02-17 07:24:26.321000+00:00
28045        E5        124.0 2026-02-17 08:26:43.333000+00:00
28063       E10        138.0 2026-02-17 08:57:03.384000+00:00
28089        E5        124.0 2026-02-17 09:12:14.102000+00:00
28424       E10        138.0 2026-02-17 12:08:26.399000+00:00
28571        E5        124.0 2026-02-17 12:32:24.365000+00:00
28600       E10        138.0 2026-02-17 12:40:53.969000+00:00
28666        E5        124.0 2026-02-17 13:08:41.712000+00:00
28667       E10        138.0 2026-02-17 13:08:41.712000+00:00
29228        E5        124.0 2026-02-17 16:48:02.435000+00:00
29229       E10        138.0 2026-02-17 16:48:02.435000+00:00
29300        E5        124.0 2026-02-17 18:56:08.722000+00:00
29604       E10        138.0 2026-02-18 09:52:08.104000+00:00
29750        E5        124.0 2026-02-18 12:00:50.487000+00:00
29906       E10        138.0 2026-02-18 14:08:19.627000+00:00


In [31]:
#chekcing the bad prices happen at a specific time period bela 
station_data_copy = station_data.copy()
station_data_copy['is_bad'] = station_data_copy['price_pence'] >500
print(station_data_copy.groupby('is_bad')['recorded_at'].agg(['min','max']))

                                    min                              max
is_bad                                                                  
False  2026-02-17 07:24:26.321000+00:00 2026-03-22 09:48:04.096000+00:00
True   2026-02-07 21:40:02.318000+00:00 2026-03-20 15:56:13.631000+00:00


so, bad priceing was started form feb 7 and good price started from feb 17 after 10 days only . this station was transmitting bad dataq from day one . then at some point started sending some good prices too . 

In [32]:
# checking the cause and history over time 

problem_station = 'c9aa1695a016aa511c09299537f3c6536c428a1958eacaa21f2b545f455ecdb5'
station_data = data[data['node_id']==problem_station]
station_data = station_data.sort_values('recorded_at')
print(station_data[['fuel_type','price_pence','recorded_at']].to_string())

          fuel_type  price_pence                      recorded_at
8506             E5       1000.0 2026-02-07 21:40:02.318000+00:00
8507            E10       1000.0 2026-02-07 21:40:02.318000+00:00
27920            E5        124.0 2026-02-17 07:24:26.321000+00:00
28018            E5       1000.0 2026-02-17 08:10:32.378000+00:00
28045            E5        124.0 2026-02-17 08:26:43.333000+00:00
28062            E5       1000.0 2026-02-17 08:57:03.384000+00:00
28063           E10        138.0 2026-02-17 08:57:03.384000+00:00
28078           E10       1000.0 2026-02-17 09:10:02.240000+00:00
28089            E5        124.0 2026-02-17 09:12:14.102000+00:00
28380            E5       1000.0 2026-02-17 11:52:34.734000+00:00
28424           E10        138.0 2026-02-17 12:08:26.399000+00:00
28473           E10       1000.0 2026-02-17 12:16:38.550000+00:00
28571            E5        124.0 2026-02-17 12:32:24.365000+00:00
28599            E5       1000.0 2026-02-17 12:40:53.969000+00:00
28600     

it seems that price are not random garbage . they are alternative 1000/999 and perfectly normal price 123penny and 137 penny . sometime s within minutes of each other . and at the end of data at march 16 and march 20 suddendly new brand fuel type appears B7_STANDARD with perfect price . 